In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib

data = {
    "age":          [25, 45, 35, 50, 23, 40, 60, 28, 33, 55],
    "income":       [30000, 80000, 50000, 120000, 25000, 70000, 95000, 40000, 60000, 110000],
    "visits":       [1, 5, 3, 8, 1, 4, 6, 2, 3, 7],
    "time_on_site": [2, 15, 8, 20, 1, 12, 18, 5, 9, 17],
    "will_buy":     [0, 1, 0, 1, 0, 1, 1, 0, 0, 1]
}

df = pd.DataFrame(data)
X = df.drop("will_buy", axis=1)
y = df["will_buy"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

joblib.dump(model, "purchase_model.pkl")
print("Model trained and saved!")

Model trained and saved!


In [2]:
import os
print(os.path.exists("purchase_model.pkl"))

True


In [3]:
!pip install fastapi uvicorn nest-asyncio pyngrok


In [6]:
# Write main.py file
main_py = """
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

model = joblib.load("purchase_model.pkl")

app = FastAPI(title="Customer Purchase Prediction")

class CustomerFeatures(BaseModel):
    age: int
    income: float
    visits: int
    time_on_site: float

@app.get("/")
def home():
    return {"message": "API is running!"}

@app.post("/predict")
def predict(customer: CustomerFeatures):
    features = np.array([[
        customer.age,
        customer.income,
        customer.visits,
        customer.time_on_site
    ]])
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0][1]

    if probability >= 0.75:
        confidence = "High"
    elif probability >= 0.50:
        confidence = "Medium"
    else:
        confidence = "Low"

    return {
        "will_buy": bool(prediction),
        "probability": round(float(probability), 4),
        "confidence": confidence,
        "message": "Customer will BUY!" if prediction == 1 else "Customer will NOT buy."
    }
"""

with open("main.py", "w") as f:
    f.write(main_py)

print("main.py created!")

main.py created!


In [7]:
import requests
import threading
import uvicorn
import nest_asyncio

nest_asyncio.apply()

# Test directly without ngrok
url = "http://127.0.0.1:8000/predict"

# Test customer 1 - Likely to BUY
customer1 = {
    "age": 45,
    "income": 80000,
    "visits": 5,
    "time_on_site": 15
}

# Test customer 2 - NOT likely to buy
customer2 = {
    "age": 23,
    "income": 25000,
    "visits": 1,
    "time_on_site": 2
}

# Test customer 3
customer3 = {
    "age": 35,
    "income": 60000,
    "visits": 3,
    "time_on_site": 9
}

# Run predictions
for i, customer in enumerate([customer1, customer2, customer3], 1):
    response = requests.post(url, json=customer)
    result = response.json()
    print(f"\n👤 Customer {i}:")
    print(f"   Age: {customer['age']}, Income: {customer['income']}")
    print(f"   Will Buy: {result['will_buy']}")
    print(f"   Probability: {result['probability']}")
    print(f"   Confidence: {result['confidence']}")
    print(f"   Message: {result['message']}")

INFO:     127.0.0.1:41218 - "POST /predict HTTP/1.1" 200 OK

👤 Customer 1:
   Age: 45, Income: 80000
   Will Buy: True
   Probability: 0.95
   Confidence: High
   Message: Customer will BUY!
INFO:     127.0.0.1:41228 - "POST /predict HTTP/1.1" 200 OK

👤 Customer 2:
   Age: 23, Income: 25000
   Will Buy: False
   Probability: 0.0
   Confidence: Low
   Message: Customer will NOT buy.
INFO:     127.0.0.1:41238 - "POST /predict HTTP/1.1" 200 OK

👤 Customer 3:
   Age: 35, Income: 60000
   Will Buy: False
   Probability: 0.18
   Confidence: Low
   Message: Customer will NOT buy.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local

In [8]:
!pip install gradio

In [9]:
import gradio as gr
import requests

def predict_customer(age, income, visits, time_on_site):
    customer = {
        "age": int(age),
        "income": float(income),
        "visits": int(visits),
        "time_on_site": float(time_on_site)
    }
    response = requests.post("http://127.0.0.1:8000/predict", json=customer)
    result = response.json()
    return (
        f"Will Buy: {result['will_buy']}",
        f"Probability: {result['probability']}",
        f"Confidence: {result['confidence']}",
        f"Message: {result['message']}"
    )

demo = gr.Interface(
    fn=predict_customer,
    inputs=[
        gr.Number(label="Age", value=35),
        gr.Number(label="Income", value=50000),
        gr.Number(label="Visits", value=3),
        gr.Number(label="Time on Site (mins)", value=10)
    ],
    outputs=[
        gr.Text(label="Will Buy?"),
        gr.Text(label="Probability"),
        gr.Text(label="Confidence"),
        gr.Text(label="Message")
    ],
    title="Customer Purchase Predictor"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://be344779d12b546b7e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
